# Walkthrough: dữ liệu, đánh giá và web

Notebook đọc kết quả đã sinh bởi package `hanoi_rentals`. Chạy từ thư mục project hoặc `notebooks/`. Để dùng notebook, cài JupyterLab riêng và chọn kernel venv. Notebook không tự crawl, train hoặc thay champion. Phạm vi chỉ căn hộ và phòng trọ.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / "data/processed/listings.csv").exists():
    ROOT = ROOT.parent
raw = pd.read_csv(ROOT / "data/processed/listings.csv")
analysis = pd.read_csv(ROOT / "data/analysis/listings_analysis.csv")
assert raw.listing_id.is_unique
assert set(raw.category) == {"can_ho_chung_cu", "nha_tro"}
display(raw.groupby(["source", "category"]).size().rename("n").reset_index())

## 1. Ngày và độ phủ

Ngày hiển thị có thể bị đẩy lại; không đồng nhất với ngày đăng đầu tiên. Xem coverage để biết lỗi nguồn và ranh giới phân trang.

In [ ]:
crawl = json.loads((ROOT / "reports/crawl_latest.json").read_text())
display(pd.DataFrame(crawl["coverage"]))
raw.assign(month=raw.posted_date.str[:7]).groupby(["month", "category"]).size().unstack().plot.bar(title="Listing dates in the accumulated sample")

## 2. Chất lượng và dữ liệu thiếu

Chưa biết nội thất khác nhà trống; thiếu diện tích không tự biến thành 0. Các quyết định review được lưu có ngữ cảnh.

In [ ]:
display(analysis.groupby(["source", "category"]).agg(n=("listing_id", "size"), eligible=("include_price", "sum"), missing_area=("area_analysis_m2", lambda s: s.isna().mean())))
display(analysis.loc[~analysis.include_price, ["listing_id", "title", "exclusion_reason"]].head(10))

## 3. Không để nhóm benchmark vào training

Benchmark này đã được xem/tái dùng; cách ly khỏi training không biến nó thành external holdout mới.

In [ ]:
split = pd.read_csv(ROOT / "data/analysis/modeling_splits.csv")
benchmark = set(pd.read_csv(ROOT / "data/benchmark/ids.csv").listing_id)
assert set(split.loc[split.partition.eq("internal_test"), "listing_id"]) == benchmark
assert set(split.loc[split.partition.eq("development"), "split_group"]).isdisjoint(set(split.loc[split.partition.eq("internal_test"), "split_group"]))
display(split.groupby(["partition", "category"]).size().rename("n"))

## 4. Candidate và champion

Báo cáo huấn luyện là candidate mới nhất. API phục vụ champion trong registry. MAE là VND/tháng; R² không phải phần trăm đúng.

In [ ]:
comparison = pd.read_csv(ROOT / "reports/model_comparison.csv")
display(comparison.loc[comparison.partition.eq("internal_test"), ["model", "mae_vnd", "rmse_vnd", "r2", "selected"]])
registry = json.loads((ROOT / "artifacts/registry.json").read_text())
print("Approved champion:", registry["champion"])
display(json.loads((ROOT / "reports/pipeline_latest.json").read_text())["evaluation"])

## 5. Kiểm tra giao diện dự đoán

Đây là một prediction minh hoạ, không phải phép đánh giá mới. Web yêu cầu thêm area_m2 ở API và chỉ chấp nhận quận/huyện trong danh sách.

In [ ]:
import joblib
from hanoi_rentals.registry import champion_directory
model = joblib.load(champion_directory(ROOT) / "model.joblib")
model.predict_listing(category="can_ho_chung_cu", district="Cầu Giấy", area=60, bedrooms=2, furnishing="Furnished")

## 6. Chạy web và demo MLOps

Từ root dùng `./scripts/run_local.sh`; API docs ở `http://127.0.0.1:8000/docs`. Đọc `docs/TEAM_PRESENTATION_GUIDE.md` cho 5 vai trò, `docs/DEPLOYMENT.md` cho GitHub Actions. Chưa có VPS/domain thì chỉ xác nhận local, không nói đã deploy production.